<a href="https://colab.research.google.com/github/iqbaal24/ML2026/blob/main/Jobsheet_Metode_Klasterisasi_(ANN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laporan Praktikum Approximate Nearest Neighbor (ANN)
Annoy, FAISS, dan HNSW dibandingkan dengan Exact Nearest Neighbor (brute force).

Cara pakai: jalankan sel dari atas ke bawah di Google Colab. Semua tabel hasil percobaan dibuat otomatis oleh kode, jadi angkanya akan muncul sendiri sesuai komputer yang dipakai. Kalau RAM Colab penuh saat memakai 1 juta data, kecilkan nilai `N_BESAR` di sel pengaturan.

In [ ]:
!pip install -q annoy faiss-cpu hnswlib

In [ ]:
import os, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss, hnswlib
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)

N_KECIL = 1_000
N_SEDANG = 100_000
N_BESAR = 1_000_000   # kecilkan (misal 200_000) jika RAM tidak cukup

def normalize(X):
    return (X / np.linalg.norm(X, axis=1, keepdims=True)).astype('float32')

def recall(I_pred, I_gt):
    """Rata-rata berapa persen tetangga yang sama dengan hasil exact."""
    I_pred, I_gt = np.atleast_2d(I_pred), np.atleast_2d(I_gt)
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I_pred, I_gt)]))

# Praktikum 1: Exact NN vs Annoy

Percobaan 1 membandingkan pencarian exact (brute force) dengan Annoy pada data random 2D. Fungsi di bawah dibuat supaya bisa dipakai untuk semua baris tabel (metric Euclidean atau Angular, jumlah tree 3 atau 8, jumlah data 1000 atau 100.000). Waktu pencarian dirata-rata dari 20 kali pengulangan supaya tidak terlalu acak. Kolom `recall` menunjukkan berapa dari 3 tetangga yang sama dengan hasil exact.

In [ ]:
def exact_numpy(X, q, k, metric):
    if metric == 'euclidean':
        d = np.linalg.norm(X - q, axis=1)
    else:  # angular = jarak cosine
        d = 1 - (X @ q) / (np.linalg.norm(X, axis=1) * np.linalg.norm(q))
    return np.argsort(d)[:k]

def run_p1(metric, trees, n, d=2, k=3, seed=42, ulang=20):
    np.random.seed(seed)
    X = (np.random.rand(n, d) * 100).astype('float32')
    q = X[np.random.randint(0, n)]

    t0 = time.perf_counter()
    for _ in range(ulang):
        idx_e = exact_numpy(X, q, k, metric)
    t_exact = (time.perf_counter() - t0) / ulang * 1000

    t0 = time.perf_counter()
    ann = AnnoyIndex(d, metric)
    for i, v in enumerate(X):
        ann.add_item(i, v)
    ann.build(trees)
    t_build = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    for _ in range(ulang):
        idx_a = ann.get_nns_by_vector(q, k)
    t_ann = (time.perf_counter() - t0) / ulang * 1000

    return {'Distance Metrics': metric.capitalize(), 'Tree': trees, 'Jumlah data': n,
            'Index ENN': list(idx_e), 'Index ANN': idx_a,
            'Waktu ENN (ms)': round(t_exact, 4), 'Waktu ANN (ms)': round(t_ann, 4),
            'Waktu build (ms)': round(t_build, 1), 'Recall': recall([idx_a], [idx_e])}

konfigurasi = [('euclidean', 3, N_KECIL), ('euclidean', 8, N_KECIL), ('euclidean', 3, N_SEDANG),
               ('angular', 3, N_KECIL), ('angular', 8, N_KECIL), ('angular', 3, N_SEDANG)]
tabel_p1 = pd.DataFrame([run_p1(*c) for c in konfigurasi])
tabel_p1

## Simulasi rekomendasi Spotify (1 juta lagu, 20 fitur)

Di modul tertulis 50 juta track, tetapi judul simulasinya 1 juta lagu dan 50 juta akan membuat RAM Colab habis (sekitar 4 GB hanya untuk datanya, belum indeksnya). Jadi dipakai 1 juta track, 20 fitur, 5 tetangga, 8 tree, sesuai tabel di modul.

In [ ]:
tabel_spotify = pd.DataFrame([run_p1(m, 8, N_BESAR, d=20, k=5, ulang=3) for m in ['euclidean', 'angular']])
tabel_spotify

**Jawaban: kenapa build index tidak dihitung waktunya?**

Karena build index hanya dilakukan satu kali di awal (offline), lalu indeks yang sudah jadi dipakai untuk melayani jutaan pencarian. Yang dirasakan pengguna aplikasi adalah kecepatan satu kali pencarian, bukan lamanya pembuatan indeks. Jadi untuk membandingkan "seberapa cepat mencari", yang adil diukur adalah waktu query saja. Meski begitu, waktu build tetap penting dipertimbangkan kalau datanya sering berubah, dan karena itu tabel di atas juga mencatat kolom `Waktu build (ms)`. Sebagai pembanding, exact NN tidak punya tahap build sehingga seluruh biayanya ada di setiap pencarian.

**Analisa singkat Praktikum 1.** Pada data kecil (1000 titik), selisih waktu exact dan Annoy sangat kecil karena menghitung 1000 jarak itu sendiri sudah sangat cepat. Keuntungan Annoy baru terlihat jelas pada data besar (100.000 dan 1 juta), ketika exact harus menghitung jarak ke semua titik sedangkan Annoy hanya memeriksa sebagian kecil titik. Menambah jumlah tree (3 menjadi 8) biasanya menaikkan recall karena pencarian melihat lebih banyak kandidat, tetapi waktu build dan memori ikut naik. Metric Angular mengukur sudut antar vektor, bukan jarak lurus, sehingga hasil tetangganya bisa berbeda dari Euclidean. Karena Annoy bersifat perkiraan, hasilnya tidak selalu sama dengan exact (lihat kolom recall).

# Praktikum 2: FAISS (IVF + PQ)

Percobaan ini membandingkan `IndexFlatL2/IP` (exact) dengan `IndexIVFPQ` (approximate). Variasinya: metric Euclidean vs Cosine, jumlah data 1000 vs 1 juta, dan dimensi 2D vs 5D. Untuk metric cosine, vektor dinormalisasi lebih dulu lalu memakai inner product. Parameter `nprobe` (jumlah cluster yang diperiksa) diset 5 supaya hasilnya tidak terlalu buruk. Pencarian memakai 100 query dan waktunya dirata-rata per query.

In [ ]:
def run_faiss(metric, n, d, k=3, nq=100, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.random((n, d), dtype=np.float32)
    Q = rng.random((nq, d), dtype=np.float32)
    if metric == 'cosine':
        X, Q = normalize(X), normalize(Q)
        flat = faiss.IndexFlatIP(d); quant = faiss.IndexFlatIP(d); mt = faiss.METRIC_INNER_PRODUCT
    else:
        flat = faiss.IndexFlatL2(d); quant = faiss.IndexFlatL2(d); mt = faiss.METRIC_L2
    nlist = 10 if n <= 1000 else 100
    m = d  # jumlah subvector harus membagi dimensi
    flat.add(X)
    t0 = time.perf_counter(); _, I_e = flat.search(Q, k); t_e = (time.perf_counter() - t0) / nq * 1000

    t0 = time.perf_counter()
    ivf = faiss.IndexIVFPQ(quant, d, nlist, m, 8, mt)
    ivf.train(X); ivf.add(X); ivf.nprobe = 5
    t_build = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); _, I_a = ivf.search(Q, k); t_a = (time.perf_counter() - t0) / nq * 1000

    return {'Metric': metric, 'Jumlah data': n, 'Dimensi': d,
            'Index ENN (query 1)': list(I_e[0]), 'Index IVF+PQ (query 1)': list(I_a[0]),
            'Waktu ENN (ms/query)': round(t_e, 5), 'Waktu IVF+PQ (ms/query)': round(t_a, 5),
            'Waktu build (ms)': round(t_build, 1), 'Recall': round(recall(I_a, I_e), 3)}

hasil = []
for metric in ['euclidean', 'cosine']:
    for n in [N_KECIL, N_BESAR]:
        for d in [2, 5]:
            hasil.append(run_faiss(metric, n, d))
tabel_p2 = pd.DataFrame(hasil)
tabel_p2

**Analisa singkat Praktikum 2.** IVF membagi data menjadi beberapa cluster lalu hanya mencari di beberapa cluster terdekat, sedangkan PQ memampatkan vektor menjadi kode kecil. Dua hal ini membuat pencarian hemat waktu dan memori, tetapi hasilnya hanya perkiraan sehingga recall bisa di bawah 1. Pada data 1000 titik, versi exact sudah sangat cepat, jadi IVF+PQ tidak terlalu menguntungkan. Pada 1 juta data, selisih waktu mulai terlihat. Pada dimensi sangat rendah (2D atau 5D), PQ kurang cocok karena tidak banyak yang bisa dimampatkan, sehingga recall bisa turun dibanding data berdimensi tinggi. Ganti metric dari Euclidean ke Cosine mengubah ukuran kemiripan, jadi tetangga terdekatnya pun bisa berbeda.

# Praktikum 3: HNSW

Perbandingan exact NN (scikit-learn brute force) dengan HNSW, dengan variasi metric (Euclidean `l2` vs `cosine`), jumlah data (1000 vs 1 juta), dan dimensi (2D vs 5D). Memakai `k = 5`, `M = 16`, `ef_construction = 100`, `ef = 50`, dan 100 query.

In [ ]:
def run_hnsw(metric, n, d, k=5, nq=100, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.random((n, d), dtype=np.float32)
    Q = rng.random((nq, d), dtype=np.float32)
    skm = 'euclidean' if metric == 'euclidean' else 'cosine'
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=skm).fit(X)
    t0 = time.perf_counter(); _, I_e = nn.kneighbors(Q); t_e = (time.perf_counter() - t0) / nq * 1000

    t0 = time.perf_counter()
    p = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=d)
    p.init_index(max_elements=n, ef_construction=100, M=16)
    p.add_items(X); p.set_ef(50)
    t_build = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); I_a, _ = p.knn_query(Q, k=k); t_a = (time.perf_counter() - t0) / nq * 1000

    return {'Metric': metric, 'Jumlah data': n, 'Dimensi': d,
            'Index ENN (query 1)': list(I_e[0]), 'Index HNSW (query 1)': list(I_a[0]),
            'Waktu ENN (ms/query)': round(t_e, 5), 'Waktu HNSW (ms/query)': round(t_a, 5),
            'Waktu build (ms)': round(t_build, 1), 'Recall': round(recall(I_a, I_e), 3)}

tabel_p3 = pd.DataFrame([run_hnsw(m, n, d) for m in ['euclidean', 'cosine'] for n in [N_KECIL, N_BESAR] for d in [2, 5]])
tabel_p3

**Analisa singkat Praktikum 3.** HNSW membuat graf berlapis, mirip peta jalan tol di atas jalan biasa, sehingga pencarian bisa "melompat" cepat ke area yang dekat dengan query. Hasilnya biasanya punya recall tinggi (sering mendekati 1) dan waktu pencarian sangat kecil, bahkan di 1 juta data. Harga yang dibayar adalah waktu build dan memori yang lebih besar karena graf harus disimpan. Pada data kecil, brute force masih cukup cepat sehingga keunggulan HNSW belum terasa.

# Praktikum 4: Annoy vs FAISS vs HNSW

Ketiganya dibandingkan pada 1 juta data 5D, `k = 10`, 100 query, untuk dua metric (Euclidean dan Angular/Cosine). FAISS yang dipakai adalah `IndexFlat` (exact), jadi hasilnya dijadikan acuan (ground truth) untuk menghitung recall Annoy dan HNSW. Annoy memakai 10 tree, HNSW memakai `M = 16`, `ef_construction = 200`, `ef = 50`.

In [ ]:
def run_p4(metric, n=N_BESAR, d=5, k=10, nq=100, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.random((n, d), dtype=np.float32)
    Q = rng.random((nq, d), dtype=np.float32)
    baris = []

    # FAISS flat (exact) = ground truth
    Xf, Qf = (normalize(X), normalize(Q)) if metric == 'angular' else (X, Q)
    t0 = time.perf_counter()
    fl = faiss.IndexFlatIP(d) if metric == 'angular' else faiss.IndexFlatL2(d)
    fl.add(Xf); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); _, I_gt = fl.search(Qf, k); tq = (time.perf_counter() - t0) / nq
    baris.append(('FAISS Flat (exact)', tb, tq, 1.0))

    # Annoy
    t0 = time.perf_counter()
    an = AnnoyIndex(d, metric)
    for i in range(n):
        an.add_item(i, X[i])
    an.build(10); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); I_an = [an.get_nns_by_vector(q, k) for q in Q]; tq = (time.perf_counter() - t0) / nq
    baris.append(('Annoy (10 tree)', tb, tq, recall(I_an, I_gt)))

    # HNSW
    t0 = time.perf_counter()
    hn = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=d)
    hn.init_index(max_elements=n, ef_construction=200, M=16)
    hn.add_items(X); hn.set_ef(50); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); I_hn, _ = hn.knn_query(Q, k=k); tq = (time.perf_counter() - t0) / nq
    baris.append(('HNSW (M=16, ef=50)', tb, tq, recall(I_hn, I_gt)))

    return pd.DataFrame([{'Metric': metric, 'Model': m, 'Build (detik)': round(b, 3),
                          'Query (ms/query)': round(q * 1000, 5), 'Recall@10': round(r, 3)}
                         for m, b, q, r in baris])

tabel_p4 = pd.concat([run_p4('euclidean'), run_p4('angular')], ignore_index=True)
tabel_p4

**Analisa singkat Praktikum 4.** Di sini terlihat kompromi tiga hal: waktu build, kecepatan query, dan akurasi. FAISS Flat tidak perlu proses build yang berat dan hasilnya selalu tepat (recall 1), tetapi ia tetap memeriksa semua data sehingga kecepatan querynya turun saat data membesar. Annoy dan HNSW butuh waktu build lebih lama, namun query-nya jauh lebih cepat. Annoy biasanya lebih ringan di memori dan mudah disimpan ke file, sedangkan HNSW biasanya memberi recall yang lebih tinggi pada kecepatan yang mirip. Perlu diingat bahwa dimensi di sini hanya 5, jadi brute force dengan FAISS masih sangat cepat, dan keunggulan ANN akan makin jelas pada dimensi yang lebih tinggi.

# Praktikum 5: Grafik Recall vs QPS

Mirip ANN-Benchmarks: sumbu X adalah recall dan sumbu Y adalah QPS (queries per second, makin tinggi makin cepat). Posisi kanan atas adalah yang terbaik. Dibandingkan dengan kode asli di modul, indeks dibangun satu kali saja lalu hanya parameter pencarian yang diubah (`search_k`, `nprobe`, `ef`), jadi jauh lebih cepat dijalankan. Data: 100.000 vektor 128 dimensi, 1000 query, `k = 10`.

In [ ]:
d, nb, nq, k = 128, N_SEDANG, 1000, 10
np.random.seed(42)
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')

flat = faiss.IndexFlatL2(d); flat.add(xb)
_, gt_idx = flat.search(xq, k)

results = {'Annoy': [], 'FAISS IVF': [], 'HNSW': []}

# Annoy: bangun sekali
annoy = AnnoyIndex(d, 'euclidean')
for i, v in enumerate(xb):
    annoy.add_item(i, v)
annoy.build(10)
for sk in [200, 500, 1000, 2000, 5000]:
    t0 = time.perf_counter()
    I = [annoy.get_nns_by_vector(v, k, search_k=sk) for v in xq]
    t = time.perf_counter() - t0
    results['Annoy'].append((recall(I, gt_idx), nq / t))

# FAISS IVF: train sekali
ivf = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, 100, faiss.METRIC_L2)
ivf.train(xb); ivf.add(xb)
for npb in [1, 5, 10, 20, 50]:
    ivf.nprobe = npb
    t0 = time.perf_counter(); _, I = ivf.search(xq, k); t = time.perf_counter() - t0
    results['FAISS IVF'].append((recall(I, gt_idx), nq / t))

# HNSW: bangun sekali
hn = hnswlib.Index(space='l2', dim=d)
hn.init_index(max_elements=nb, ef_construction=200, M=16)
hn.add_items(xb)
for ef in [20, 50, 100, 200, 400]:
    hn.set_ef(ef)
    t0 = time.perf_counter(); I, _ = hn.knn_query(xq, k); t = time.perf_counter() - t0
    results['HNSW'].append((recall(I, gt_idx), nq / t))

plt.figure(figsize=(8, 6))
for (label, pts), c in zip(results.items(), ['blue', 'green', 'red']):
    r, qps = zip(*pts)
    plt.plot(r, qps, marker='o', label=label, color=c)
plt.xlabel('Recall@10'); plt.ylabel('Queries per second (log)')
plt.yscale('log'); plt.title('ANN Benchmark: Annoy vs FAISS IVF vs HNSW')
plt.legend(); plt.grid(True, which='both', ls='--', alpha=0.6)
plt.show()

pd.DataFrame([(m, round(r, 3), round(q, 1)) for m, pts in results.items() for r, q in pts],
             columns=['Algoritma', 'Recall', 'QPS'])

**Analisa singkat Praktikum 5.** Setiap garis bergerak dari kiri atas ke kanan bawah: saat parameter pencarian dinaikkan, recall naik tetapi QPS turun. Itulah trade-off akurasi dan kecepatan. Algoritma yang kurvanya paling dekat ke pojok kanan atas adalah yang terbaik. Catatan penting: data random berdimensi 128 adalah kasus sulit untuk semua ANN karena tidak punya struktur cluster, sehingga recall bisa lebih rendah dibanding data nyata. Hasil pada data nyata (seperti data lagu di Praktikum 6) biasanya lebih baik.

# Praktikum 6: Dataset Micro Spotify

Dataset: <https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data> (pilih dataset pertama). Unduh CSV-nya lalu upload ke Colab (atau taruh di Google Drive). Hanya fitur numerik yang dipakai, lalu distandarisasi dengan `StandardScaler`.

Dibanding kode di modul, ada dua tambahan: waktu build dan waktu query dipisah (jawab pertanyaan sebelumnya), dan ada perhitungan recall supaya perbandingannya adil. `MAX_DATA` membatasi jumlah lagu agar exact NN (yang menghitung semua pasangan lagu) tidak memakan waktu terlalu lama. Isi `None` jika ingin memakai semua data.

In [ ]:
CSV_PATH = 'spotify_songs.csv'   # ganti jika file ada di Drive, mis. '/content/drive/MyDrive/.../spotify_songs.csv'
MAX_DATA = 100_000

if not os.path.exists(CSV_PATH):
    from google.colab import files
    up = files.upload()               # pilih file CSV dari komputer
    CSV_PATH = list(up.keys())[0]

df = pd.read_csv(CSV_PATH)
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
features = [f for f in features if f in df.columns]
df = df.dropna(subset=features).reset_index(drop=True)
if MAX_DATA and len(df) > MAX_DATA:
    df = df.sample(MAX_DATA, random_state=42).reset_index(drop=True)
print('Jumlah lagu:', len(df), '| Fitur:', features)

X_scaled = StandardScaler().fit_transform(df[features].values).astype('float32')
n, f = X_scaled.shape
k = 10

In [ ]:
rows = {}

# Exact NN
t0 = time.perf_counter()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean').fit(X_scaled)
tb = time.perf_counter() - t0
t0 = time.perf_counter(); _, idx_exact = nn.kneighbors(X_scaled); tq = time.perf_counter() - t0
rows['Exact NN'] = (tb, tq, 1.0)

# Annoy
t0 = time.perf_counter()
an = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    an.add_item(i, v)
an.build(10); tb = time.perf_counter() - t0
t0 = time.perf_counter(); idx_annoy = np.array([an.get_nns_by_vector(v, k) for v in X_scaled]); tq = time.perf_counter() - t0
rows['Annoy'] = (tb, tq, recall(idx_annoy, idx_exact))

# HNSW
t0 = time.perf_counter()
hn = hnswlib.Index(space='l2', dim=f)
hn.init_index(max_elements=n, ef_construction=200, M=16)
hn.add_items(X_scaled); hn.set_ef(200); tb = time.perf_counter() - t0
t0 = time.perf_counter(); idx_hnsw, _ = hn.knn_query(X_scaled, k=k); tq = time.perf_counter() - t0
rows['HNSW'] = (tb, tq, recall(idx_hnsw, idx_exact))

# FAISS IVF
t0 = time.perf_counter()
iv = faiss.IndexIVFFlat(faiss.IndexFlatL2(f), f, 100, faiss.METRIC_L2)
iv.train(X_scaled); iv.add(X_scaled); iv.nprobe = 10; tb = time.perf_counter() - t0
t0 = time.perf_counter(); _, idx_faiss = iv.search(X_scaled, k); tq = time.perf_counter() - t0
rows['FAISS IVF'] = (tb, tq, recall(idx_faiss, idx_exact))

tabel_p6 = pd.DataFrame([(m, round(b, 3), round(q, 3), round(b + q, 3), round(r, 3)) for m, (b, q, r) in rows.items()],
                        columns=['Model', 'Build (s)', 'Query semua lagu (s)', 'Total (s)', 'Recall@10'])
print(tabel_p6.to_string(index=False))

print('\nTop-5 tetangga lagu pertama:')
for nama, idx in [('Exact', idx_exact), ('Annoy', idx_annoy), ('HNSW', idx_hnsw), ('FAISS', idx_faiss)]:
    print(f'{nama:6s}:', idx[0][:5])

# Contoh judul lagu (jika ada kolom nama)
kol_nama = [c for c in ['name', 'track_name', 'title'] if c in df.columns]
if kol_nama:
    print('\nLagu acuan:', df.loc[0, kol_nama[0]])
    print('Mirip (Exact):', list(df.loc[idx_exact[0][1:6], kol_nama[0]]))

**Analisa Praktikum 6.** Kode ini mencari 10 lagu paling mirip untuk setiap lagu, berdasarkan fitur audio (danceability, energy, loudness, dan seterusnya) yang sudah distandarisasi. Standarisasi penting karena fitur seperti `tempo` dan `loudness` punya rentang angka jauh lebih besar dari fitur lain, sehingga tanpa standarisasi jarak akan didominasi fitur tersebut.

Dari tabel hasil, bandingkan dua hal: waktu total dan recall. Exact NN adalah acuan yang paling benar (recall 1), tetapi paling lambat karena menghitung jarak setiap lagu ke semua lagu lain. Annoy, HNSW, dan FAISS IVF umumnya jauh lebih cepat. HNSW biasanya memberi recall tertinggi di antara ketiganya, tetapi waktu build-nya lebih lama. FAISS IVF sangat cepat saat query, namun recall-nya bergantung pada `nprobe`. Annoy ada di tengah dan recall-nya bergantung pada jumlah tree.

Karena fitur lagu hanya 9 dimensi dan datanya tidak sebesar jutaan, selisih kecepatan dengan exact tidak terlalu dramatis. Untuk sistem rekomendasi besar seperti Spotify yang punya ratusan juta lagu, perbedaannya menjadi sangat besar. Hasil ANN adalah perkiraan, jadi beberapa tetangga bisa berbeda dari hasil exact, tetapi untuk rekomendasi musik hal ini bisa diterima karena lagu yang tertukar umumnya tetap mirip.

# Tugas: Google Colab vs PyDroid3 (Android)

Jalankan seluruh sel Tugas di bawah ini di Google Colab, lalu salin kode yang sama ke aplikasi PyDroid3 di HP dan jalankan di sana. Di PyDroid3, install dulu lewat menu Pip: `numpy`, `scikit-learn`, `annoy`, `hnswlib`, dan `faiss-cpu` (jika `faiss-cpu` atau `annoy` gagal terpasang di Android, catat sebagai temuan di analisa). Kode di bawah sudah ditambah perhitungan recall dan pencetak spesifikasi perangkat. Sel spesifikasi hanya mengandalkan modul bawaan Python supaya tetap jalan di PyDroid3.

In [ ]:
import numpy as np, time, platform, os
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib, faiss
from sklearn.preprocessing import StandardScaler

# ---- Spesifikasi perangkat ----
print('Platform :', platform.platform())
print('Prosesor :', platform.processor() or platform.machine())
print('Jumlah core CPU :', os.cpu_count())
try:
    import psutil
    print('RAM :', round(psutil.virtual_memory().total / 1e9, 2), 'GB')
except Exception:
    try:
        print('RAM :', open('/proc/meminfo').readline().strip())
    except Exception:
        print('RAM : isi manual')
print('Python :', platform.python_version())
print('-' * 40)

np.random.seed(42)
n_samples, d, k = 10000, 128, 10
X = np.random.random((n_samples, d)).astype('float32')
X_scaled = StandardScaler().fit_transform(X).astype('float32')

def rec(pred, gt):
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(pred, gt)])

# Exact NN
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean').fit(X_scaled)
_, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# Annoy
start = time.time()
ai = AnnoyIndex(d, 'euclidean')
for i, v in enumerate(X_scaled):
    ai.add_item(i, v)
ai.build(10)
idx_annoy = [ai.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# HNSW
start = time.time()
hn = hnswlib.Index(space='l2', dim=d)
hn.init_index(max_elements=n_samples, ef_construction=200, M=16)
hn.add_items(X_scaled); hn.set_ef(200)
idx_hnsw, _ = hn.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# FAISS IVF
start = time.time()
iv = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, 100, faiss.METRIC_L2)
iv.train(X_scaled); iv.add(X_scaled); iv.nprobe = 10
_, idx_faiss = iv.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

print("\n=== Ringkasan (detik) dan Recall@10 ===")
print(f"Exact NN : {time_exact:.3f}  (recall 1.000)")
print(f"Annoy    : {time_annoy:.3f}  (recall {rec(idx_annoy, idx_exact):.3f})")
print(f"HNSW     : {time_hnsw:.3f}  (recall {rec(idx_hnsw, idx_exact):.3f})")
print(f"FAISS    : {time_faiss:.3f}  (recall {rec(idx_faiss, idx_exact):.3f})")

## Spesifikasi perangkat dan hasil

Spesifikasi smartphone yang dipakai: **(isi: merek/tipe HP, chipset, jumlah core, RAM, versi Android)**.

Spesifikasi Colab: lihat keluaran sel di atas (CPU, jumlah core, RAM).

Tabel perbandingan (isi dari keluaran masing-masing perangkat):

| Metode | Colab (detik) | PyDroid3 (detik) | Recall |
|---|---|---|---|
| Exact NN | ... | ... | 1.000 |
| Annoy | ... | ... | ... |
| HNSW | ... | ... | ... |
| FAISS IVF | ... | ... | ... |

## Analisa

Secara umum Colab akan lebih cepat daripada HP karena CPU server punya lebih banyak core, memori lebih besar, dan library seperti scikit-learn dan FAISS memakai instruksi yang dioptimalkan (BLAS dan SIMD). Prosesor HP dirancang hemat daya, sehingga kecepatan bisa turun saat HP panas atau baterai hemat daya aktif. Karena itu, selisih waktu paling terasa pada metode yang berat di hitungan, yaitu Exact NN dan FAISS.

Urutan antar metode biasanya tetap sama di kedua perangkat: Exact NN paling lambat, sedangkan ANN (Annoy, HNSW, FAISS IVF) lebih cepat dengan recall yang sedikit di bawah 1. Artinya keuntungan ANN tidak bergantung pada perangkat, hanya angka detiknya yang berbeda. Perlu diingat bahwa waktu yang diukur di sini sudah termasuk build index, jadi metode seperti HNSW yang build-nya berat bisa terlihat kurang unggul pada data hanya 10.000 titik.

Catatan lain: data random 128 dimensi tidak punya struktur, sehingga recall ANN lebih rendah dibanding data nyata. Di PyDroid3 beberapa library (terutama `faiss-cpu` dan `annoy`) kadang sulit dipasang karena perlu dikompilasi untuk Android. Jika ada yang gagal, tuliskan itu sebagai keterbatasan, dan bandingkan hanya metode yang berhasil dijalankan.

**(Ganti angka dan kalimat di atas dengan hasil yang kamu dapat dari kedua perangkat.)**